# 04 · Prepare real context and independent reference frames

**Goal:** define recording-disjoint GAVD collections and prepare a
human annotation template without creating pose pseudolabels.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](README.md)
· [HAIC setup and launch commands](../../slurm/synthetic-training/README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

## 1. Select recordings using visible scene information

GAVD context, early evaluation, and confirmation recordings must be
separate. Existing study reservations remain excluded. Recording
independence does not establish that every person is unique.

The first call writes a view/crop template when checked metadata is
missing. Review each proposed recording's coarse view, source-person
height (`person_height_px`), and fixed person crop. Use `side`, `oblique`, or `frontal_rear`
and explicitly mark `view_checked=true`. Full video height is not the
person's source resolution. Set `ST_GAVD_VIEWS` to the completed CSV,
restart this notebook, and rerun the preparation stage.

In [ ]:
prepared = workflow.prepare_gavd(cfg)
show_result(prepared)

## 2. Understand the two real-data roles

Context recordings supply unlabeled images, predictions, and response
information to the selector. Independent reference recordings supply
images for prediction and human labels for later scoring. The selector
never receives those human labels.

The planned groups cross three coarse views with low/high source-person
resolution. Per setting, the defaults are three context recordings,
four early references, and six confirmation references. These are
requested counts, not guaranteed available data. Do not silently reuse
a recording if a group is too small.

In [ ]:
for name in ("gavd_context.csv", "gavd_evaluation.csv"):
    path = RUN_ROOT / "data" / name
    if path.is_file():
        table = pd.read_csv(path)
        display(Markdown(f"### {name}"))
        display(table.head(8))
        groups = [c for c in ("role", "domain_id") if c in table]
        if groups and "recording_id" in table:
            display(table.groupby(groups).recording_id.nunique().rename("recordings").to_frame())

## 3. Annotate references independently

Open `data/annotate-early.html` or `data/annotate-confirmation.html` in
a local browser after copying the run's `data` directory with its
`gavd` image tree. Each page displays the exported full-frame PNGs.
Enter the annotator name, mark the independent reference box with two
clicks, and mark each visible joint or label it hidden. Save JSON
progress between sessions, then export the completed CSV.

The page exports separate early and confirmation annotation CSVs:
`frame_id`, `landmark`, `x`, `y`, `visible`, `box_x1`, `box_y1`, `box_x2`,
`box_y2`, `annotator`, and `reviewer`. Coordinates refer to the exported
full image. Mark twelve rows per frame: left/right shoulders, elbows,
wrists, hips, knees, and ankles. For hidden or uncertain landmarks set
`visible=false` and leave coordinates blank. Do not guess them.

Independently mark the person's reference box on each frame. Its
diagonal supplies the model-independent error scale. A second reviewer
should load the saved progress, inspect each frame, and mark it
reviewed. Confirmation requires a reviewer different from its annotator.
Never initialize
the reference coordinates from one of the evaluated models.

Save completed `gavd-early-annotations.csv` and
`gavd-confirmation-annotations.csv` in one directory and point
`ST_GAVD_ANNOTATIONS` to that directory before notebook 06. A path
pattern containing `{split}` is also accepted. Evaluation opens only
the requested split; confirmation requires a second reviewer.
Predictions in notebook 05 do not require either CSV. Templates are
not valid reference data until a human completes them.

Next: [05 · Choose and adapt](05_choose_and_adapt.ipynb).